# Does Housing Supply Constrain Home Prices? Evidence from U.S. Metro Areas

### ECON 320 Lab — Final Project
**Student names:** Tessa Butler, Siya Kumar, Ania Ting, Naman Keswani, Shunji Lewandowski  
**Lab Section:** [Lab #2]  

---
## Abstract
We study whether housing supply constrains home prices across U.S. Metropolitan Statistical Areas (MSAs) using a cross-sectional OLS framework. The unit of observation is an MSA (N ≈ 919). The dependent variable is the FHFA All-Transactions House Price Index (2024, CBSA-level). The primary regressor is the number of new residential units authorized by building permit (Census BPS, January 2026 snapshot). The baseline specification is ln(HPI<sub>i</sub>) = β<sub>0</sub> + β<sub>1</sub>·ln(Permits<sub>i</sub>) + γ′Controls<sub>i</sub> + ε<sub>i</sub>. We find [main result — complete after running regressions]. Results are [robust/not robust] to [alternative specification]. We discuss omitted variable bias from demand-side factors such as income and population, and multicollinearity between supply and demand indicators. Our findings [implication for housing policy].

---
## 0. Setup

In [17]:
%pip install pandas numpy matplotlib statsmodels scipy xlrd openpyxl

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.outliers_influence import variance_inflation_factor
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.dpi'] = 130
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False


[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


---
## 1. Introduction

**Research question:** Does greater housing supply — measured by new residential building permits — reduce home prices across U.S. metropolitan areas?

**Why it matters:** The U.S. housing affordability crisis has worsened steadily. By 2025, the cumulative supply gap reached an estimated 4.03 million homes (Realtor.com, 2026), with the median down payment requiring seven years of saving for a typical household. Over 42 million Americans spend more than 30% of income on housing. Standard supply-and-demand theory predicts that constrained supply, driven by zoning restrictions, permitting delays, and construction costs, pushes prices above competitive equilibrium. Yet the empirical magnitude of this relationship across metro areas remains a policy-relevant question.

**Approach:** We exploit cross-sectional variation in building permit activity across ~919 MSAs. Using OLS with heteroskedasticity-robust standard errors (HC3), we regress log home prices on log permits, controlling for demand-side factors. The log-log specification yields elasticity estimates directly interpretable as percentage responses.

---
## 2. Data and Summary Statistics

**Data sources:**
- **FHFA All-Transactions HPI (Annual, CBSA-level):** Federal Housing Finance Agency. Repeat-sales index using Fannie Mae/Freddie Mac loan data plus FHA and county recorder data. Covers all MSAs in the U.S. We use the 2024 annual index value (base year 1990).
- **Census Building Permits Survey (BPS), CBSA Monthly:** U.S. Census Bureau. Reports new privately-owned residential units authorized by building permit. We use the January 2026 monthly release as a cross-sectional snapshot of MSA-level supply activity. *Limitation: ideally we would use 12-month totals for 2024; the January 2026 snapshot introduces measurement noise but cross-sectional MSA rankings are persistent.*
- **ACS 2024 1-Year Estimates (to be added):** Median household income (B19013), total housing units (B25001), and population (B01003) at the MSA level. **Action required — re-download from data.census.gov selecting all MSAs, not United States. Save as** `data/acs_controls.csv`.

**Unit of observation:** MSA (Metropolitan Statistical Area), identified by 5-digit CBSA code.

**Variables:**
- *y* — `log_hpi`: Natural log of FHFA All-Transactions HPI 2024 (proxy for price level)
- *x* — `log_permits`: Natural log of total residential units permitted (Jan 2026)
- *Controls (pending ACS download)*: `log_income` (ln median HH income), `log_pop` (ln population)

### 2.1 Load and Prepare Data

In [18]:
hpi_raw = pd.read_excel('data/hpi_at_cbsa.xlsx', skiprows=5, header=None)
hpi_raw.columns = ['CBSA','Name','Year','Ann_Chg_Pct','HPI','HPI_1990base','HPI_2000base']
hpi_raw['CBSA'] = pd.to_numeric(hpi_raw['CBSA'], errors='coerce')
hpi_raw['Year'] = pd.to_numeric(hpi_raw['Year'], errors='coerce')
hpi_raw['HPI'] = pd.to_numeric(hpi_raw['HPI'], errors='coerce')
hpi_raw['Ann_Chg_Pct'] = pd.to_numeric(hpi_raw['Ann_Chg_Pct'], errors='coerce')

hpi = hpi_raw[(hpi_raw['CBSA'] >= 10000) & (hpi_raw['Year'] == 2024)][['CBSA','Name','HPI','Ann_Chg_Pct']].copy()
hpi['CBSA'] = hpi['CBSA'].astype(int)
hpi = hpi.dropna(subset=['HPI']).reset_index(drop=True)
print(f'FHFA HPI 2024: {len(hpi)} MSAs')
hpi.head()

FHFA HPI 2024: 920 MSAs


,CBSA,Name,HPI,Ann_Chg_Pct
0,10100,"Aberdeen, SD",635.69,-0.15
1,10140,"Aberdeen, WA",994.85,3.21
2,10180,"Abilene, TX",550.53,3.63
3,10220,"Ada, OK",312.94,2.29
4,10300,"Adrian, MI",554.87,5.12


In [19]:
perm_raw = pd.read_excel('data/cbsamonthly_202601.xls', engine='xlrd', skiprows=6, header=None)
data_rows = perm_raw.iloc[2:].reset_index(drop=True)

perm = pd.DataFrame({
    'CBSA': pd.to_numeric(data_rows[1], errors='coerce'),
    'total_permits': pd.to_numeric(data_rows[11], errors='coerce')
})
perm = perm.dropna(subset=['CBSA','total_permits'])
perm['CBSA'] = perm['CBSA'].astype(int)
perm = perm.reset_index(drop=True)
print(f'BPS permits: {len(perm)} MSAs')
perm.head()

BPS permits: 921 MSAs


,CBSA,total_permits
0,10100,1
1,10140,22
2,10180,114
3,10220,0
4,10300,12


In [20]:
df = hpi.merge(perm, on='CBSA', how='inner')
print(f'Merged: {df.shape[0]} MSAs')
df.head()

Merged: 917 MSAs


,CBSA,Name,HPI,Ann_Chg_Pct,total_permits
0,10100,"Aberdeen, SD",635.69,-0.15,1
1,10140,"Aberdeen, WA",994.85,3.21,22
2,10180,"Abilene, TX",550.53,3.63,114
3,10220,"Ada, OK",312.94,2.29,0
4,10300,"Adrian, MI",554.87,5.12,12


In [21]:
df['log_hpi']     = np.log(df['HPI'])
df['log_permits'] = np.log(df['total_permits'].replace(0, np.nan))
df = df.dropna(subset=['log_hpi','log_permits']).reset_index(drop=True)
print(f'Analysis sample (non-zero permits): {len(df)} MSAs')
print(f"Dropped (zero-permit MSAs): {hpi.merge(perm, on='CBSA').shape[0] - len(df)}")

Analysis sample (non-zero permits): 842 MSAs
Dropped (zero-permit MSAs): 75


In [22]:
#ACS Controls: Load & Merge 
def load_acs(path, val_col, new_name):
    raw = pd.read_csv(path, skiprows=[1], dtype=str)
    raw['CBSA'] = pd.to_numeric(raw['GEO_ID'].str[-5:], errors='coerce')
    raw[new_name] = pd.to_numeric(raw[val_col], errors='coerce')
    return raw[['CBSA', new_name]].dropna()

inc   = load_acs('data/Median Household Income B19013-Data.csv',  'B19013_001E', 'median_income')
pop   = load_acs('data/Total Population B01003-Data.csv',         'B01003_001E', 'population')
units = load_acs('data/Housing Units B25001-Data.csv',            'B25001_001E', 'housing_units')

acs = inc.merge(pop, on='CBSA').merge(units, on='CBSA')
acs = acs[acs['median_income'] > 0].reset_index(drop=True)
print(f'ACS controls: {len(acs)} MSAs')
acs.head()

ACS controls: 535 MSAs


,CBSA,median_income,population,housing_units
0,10140,59619,77893,37782
1,10180,63390,181969,76934
2,10300,71632,97746,44014
3,10380,24351,250969,126694
4,10420,71364,702209,321578


In [23]:
df2 = df.merge(acs, on='CBSA', how='inner')
print(f'Final sample: {len(df2)} MSAs')
print(df2.columns.tolist())
df2.head()

Final sample: 520 MSAs
['CBSA', 'Name', 'HPI', 'Ann_Chg_Pct', 'total_permits', 'log_hpi', 'log_permits', 'median_income', 'population', 'housing_units']


,CBSA,Name,HPI,Ann_Chg_Pct,total_permits,log_hpi,log_permits,median_income,population,housing_units
0,10140,"Aberdeen, WA",994.85,3.21,22,6.902592,3.091042,59619,77893,37782
1,10180,"Abilene, TX",550.53,3.63,114,6.310881,4.736198,63390,181969,76934
2,10300,"Adrian, MI",554.87,5.12,12,6.318734,2.484907,71632,97746,44014
3,10420,"Akron, OH",602.58,8.15,50,6.401220,3.912023,71364,702209,321578
4,10460,"Alamogordo, NM",495.14,10.24,1,6.204841,0.000000,56354,69711,33302


### 2.2 Descriptive Statistics

In [ ]:
desc_vars = ['HPI','Ann_Chg_Pct','total_permits','log_hpi','log_permits']
desc = df[desc_vars].describe().T.round(3)
desc.columns = ['N','Mean','Std','Min','25%','50%','75%','Max']
print(desc.to_string())

                   N     Mean      Std      Min      25%      50%      75%       Max
HPI            842.0  645.412  372.772  154.030  395.898  537.815  764.745  2883.610
Ann_Chg_Pct    842.0    5.405    3.182  -11.860    3.495    5.400    7.290    20.260
total_permits  842.0  116.262  388.869    1.000    4.000   14.000   57.000  6012.000
log_hpi        842.0    6.339    0.493    5.037    5.981    6.288    6.640     7.967
log_permits    842.0    2.862    1.846    0.000    1.386    2.639    4.043     8.702


---
## 3. Empirical Strategy

We estimate the following log-log OLS models:

**Model 1 — Bivariate (OVB benchmark):**

$$\ln(\text{HPI}_i) = \beta_0 + \beta_1 \ln(\text{Permits}_i) + \varepsilon_i$$

**Model 2 — Full specification (controls for demand-side OVB):**

$$\ln(\text{HPI}_i) = \beta_0 + \beta_1 \ln(\text{Permits}_i) + \beta_2 \ln(\text{Income}_i) + \beta_3 \ln(\text{Pop}_i) + \varepsilon_i$$

**Interpretation of β₁:** A 1% increase in permitted units is associated with a β₁% change in the HPI, holding other factors constant. Supply-demand theory predicts **β₁ < 0**: more supply → lower prices.

**Omitted Variable Bias (OVB):** The bivariate estimate of β₁ is likely upward biased (toward zero or positive) because demand-rich metros — with high income and population — simultaneously attract more construction *and* have higher prices. This positive correlation between permits and income/population biases β₁ upward in Model 1. Controlling for income and population in Model 2 should make β₁ more negative, moving it toward the true supply effect.

**Multicollinearity concern:** Log permits and log population will be positively correlated (large cities permit more units). We check Variance Inflation Factors (VIF) to ensure estimates are still interpretable.

**Estimation:** OLS with HC3 heteroskedasticity-robust standard errors throughout.

---
## References

- **FHFA House Price Index:** Federal Housing Finance Agency. *FHFA HPI All-Transactions CBSA Annual Data.* https://www.fhfa.gov/data/hpi/datasets (2025).
- **Census Building Permits Survey:** U.S. Census Bureau. *New Privately-Owned Residential Building Units Authorized, CBSA Monthly.* https://www.census.gov/construction/bps (2026).
- **ACS 2024:** U.S. Census Bureau. *American Community Survey 1-Year Estimates,* Tables B19013, B25001, B01003. https://data.census.gov (2024).
- Glaeser, E. L., Gyourko, J., & Saks, R. E. (2005). Why have housing prices gone up? *American Economic Review, 95*(2), 329–333.
- Realtor.com. (2026). *2026 Housing Supply Gap Report.*
- Wooldridge, J. M. (2019). *Introductory Econometrics: A Modern Approach* (7th ed.). Cengage.